In [1]:
!git clone -b sft_evaluation https://github.com/alapedriza1/mistral-7b-enterprise-function-calling.git

Cloning into 'mistral-7b-enterprise-function-calling'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 244 (delta 37), reused 37 (delta 19), pack-reused 167 (from 1)
Receiving objects: 100% (244/244), 832.41 KiB | 3.80 MiB/s, done.
Resolving deltas: 100% (144/144), done.


# 04 - Evaluation

Evaluates the fine-tuned adapter ([alapedriza/mistral-7b-function-calling-adapter](https://huggingface.co/alapedriza/mistral-7b-function-calling-adapter)) against the baseline Mistral-7B-Instruct-v0.3 on the 139-example test set.

Compares tool selection accuracy, JSON validity, argument matching, and no-tool restraint.

In [2]:
%pip install -q transformers accelerate bitsandbytes peft trl datasets tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 32.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import os
import json
import pandas as pd
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

PROJECT_ROOT = "/kaggle/working/mistral-7b-enterprise-function-calling"
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_jsonl
from src.inference import load_finetuned_model, run_inference_on_test_set
from src.evaluation import evaluate_results
from src.reporting import overall_summary, breakdown_by_category, breakdown_by_tool

In [4]:
DATA_DIR = f"{PROJECT_ROOT}/data"
RESULTS_DIR = f"{PROJECT_ROOT}/results"

test_data = load_jsonl(f"{DATA_DIR}/test.jsonl")
print(f"Test set: {len(test_data)} examples")

Test set: 154 examples


In [5]:
baseline_summary = pd.read_csv(f"{RESULTS_DIR}/baseline_summary.csv", index_col=0)
print(f"Loaded baseline summary from {RESULTS_DIR}/baseline_summary.csv")
baseline_summary

Loaded baseline summary from /kaggle/working/mistral-7b-enterprise-function-calling/results/baseline_summary.csv


,score
JSON Validity Rate,0.870504
Function Name Accuracy,0.917355
Required Fields Completeness,0.955372
Type Correctness Rate,1.000000
Enum Compliance Rate,1.000000
Avg Hallucinated Params,0.008264
Argument Value Match,0.768372
No-Tool Restraint Accuracy,1.000000


## Fine-Tuned Model

In [6]:
HF_REPO_ID = "alapedriza/mistral-7b-function-calling-adapter"
print(f"Loading fine-tuned model from {HF_REPO_ID}...", flush=True)
ft_model, ft_tokenizer = load_finetuned_model(HF_REPO_ID)
print("Running inference on test set...", flush=True)
finetuned_results = run_inference_on_test_set(ft_model, ft_tokenizer, test_data)

Loading fine-tuned model from alapedriza/mistral-7b-function-calling-adapter...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

Running inference on test set...



Running inference:   0%|          | 0/154 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Running inference: 100%|██████████| 154/154 [1:18:19<00:00, 30.52s/it]

Inference complete: 154 examples


In [7]:
finetuned_eval_df = evaluate_results(finetuned_results)
finetuned_summary = overall_summary(finetuned_eval_df)
finetuned_summary

,score
JSON Validity Rate,0.978417
Function Name Accuracy,0.970588
Required Fields Completeness,0.955882
Type Correctness Rate,1.000000
Enum Compliance Rate,0.992647
Avg Hallucinated Params,0.000000
Argument Value Match,0.836714
No-Tool Restraint Accuracy,1.000000


In [8]:
finetuned_cat = breakdown_by_category(finetuned_eval_df)
finetuned_cat

,json_valid,func_name_correct,required_fields,type_correct,enum_compliant,hallucinated_params,argument_match,no_tool_restraint,n
category,,,,,,,,,
ambiguous,1.0,0.9375,0.812500,1.0,1.000000,0.0,0.704018,NaN,16
complex,1.0,0.972973,0.972973,1.0,0.972973,0.0,0.775933,NaN,37
multi_tool,0.875,1.0,1.000000,1.0,1.000000,0.0,0.784249,NaN,24
no_tool,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,15
simple,1.0,0.967742,0.967742,1.0,1.000000,0.0,0.925000,NaN,62


In [9]:
finetuned_tool = breakdown_by_tool(finetuned_eval_df, finetuned_results)
finetuned_tool

,json_valid,func_name_correct,required_fields,type_correct,enum_compliant,hallucinated_params,argument_match,n
tool_name,,,,,,,,
create_audit_task,1.0,1.0,1.000000,1.0,1.000000,0.0,0.809524,6
create_invoice,0.625,1.0,1.000000,1.0,1.000000,0.0,0.960000,8
escalate_issue,1.0,1.0,1.000000,1.0,1.000000,0.0,0.787500,8
extract_document_data,1.0,0.9,0.900000,1.0,1.000000,0.0,0.845455,10
generate_report,1.0,1.0,1.000000,1.0,1.000000,0.0,0.958333,8
get_customer_risk_profile,1.0,1.0,1.000000,1.0,1.000000,0.0,1.000000,9
get_portfolio_summary,1.0,1.0,1.000000,1.0,1.000000,0.0,1.000000,2
get_transaction_history,1.0,0.909091,0.909091,1.0,1.000000,0.0,0.822222,11
get_workflow_status,1.0,1.0,1.000000,1.0,1.000000,0.0,0.929762,15


In [10]:
# Save fine-tuned results
with open(f"{RESULTS_DIR}/finetuned_results.jsonl", "w") as f:
    for r in finetuned_results:
        f.write(json.dumps(r) + "\n")

finetuned_eval_df.to_csv(f"{RESULTS_DIR}/finetuned_eval_df.csv", index=True)
finetuned_summary.to_csv(f"{RESULTS_DIR}/finetuned_summary.csv", index=True)
finetuned_cat.to_csv(f"{RESULTS_DIR}/finetuned_cat_breakdown.csv", index=True)
finetuned_tool.to_csv(f"{RESULTS_DIR}/finetuned_tool_breakdown.csv", index=True)

print(f"Results saved to {RESULTS_DIR}/")

Results saved to /kaggle/working/mistral-7b-enterprise-function-calling/results/


In [11]:
# Failure-mode breakdown: non-overlapping buckets so counts sum to total errors.

tool_examples = finetuned_eval_df[finetuned_eval_df["category"] != "no_tool"]
n_tool = len(tool_examples)

# Build a waterfall: each bucket removes examples claimed by previous ones
parseable = tool_examples[tool_examples["json_valid"] == True]
correct_func = parseable[parseable["func_name_correct"] == True]

failure_buckets = {
    "JSON Parse Failures": len(tool_examples) - len(parseable),
    "Wrong Function Name": len(parseable) - len(correct_func),
    "Missing Required Fields": (correct_func["required_fields"] < 1.0).sum() if len(correct_func) else 0,
    "Hallucinated Params": (correct_func["hallucinated_params"] > 0).sum() if len(correct_func) else 0,
}

print(f"Tool-call examples: {n_tool}\n")
for label, count in failure_buckets.items():
    print(f"  {label:.<30} {count:>4} / {n_tool}  ({count/n_tool:.1%})")

# No-tool restraint (separate population)
no_tool = finetuned_eval_df[finetuned_eval_df["category"] == "no_tool"]
if len(no_tool) > 0:
    false_triggers = (no_tool["no_tool_restraint"] == False).sum()
    print(f"\nNo-Tool False Triggers: {false_triggers} / {len(no_tool)} "
          f"({false_triggers/len(no_tool):.1%})")

Tool-call examples: 139

  JSON Parse Failures...........    3 / 139  (2.2%)
  Wrong Function Name...........    4 / 139  (2.9%)
  Missing Required Fields.......    2 / 139  (1.4%)
  Hallucinated Params...........    0 / 139  (0.0%)

No-Tool False Triggers: 0 / 15 (0.0%)


## Comparison

In [12]:
comparison = baseline_summary.rename(columns={"score": "Baseline"}).join(
    finetuned_summary.rename(columns={"score": "Fine-Tuned"})
)
comparison["Delta"] = comparison["Fine-Tuned"] - comparison["Baseline"]
comparison

,Baseline,Fine-Tuned,Delta
JSON Validity Rate,0.870504,0.978417,0.107914
Function Name Accuracy,0.917355,0.970588,0.053233
Required Fields Completeness,0.955372,0.955882,0.000510
Type Correctness Rate,1.000000,1.000000,0.000000
Enum Compliance Rate,1.000000,0.992647,-0.007353
Avg Hallucinated Params,0.008264,0.000000,-0.008264
Argument Value Match,0.768372,0.836714,0.068341
No-Tool Restraint Accuracy,1.000000,1.000000,0.000000


In [13]:
baseline_cat = pd.read_csv(f"{RESULTS_DIR}/baseline_cat_breakdown.csv", index_col=0)

# Join on shared metrics, suffix to distinguish
cat_comparison = baseline_cat.drop(columns="n").add_suffix(" (Baseline)").join(
    finetuned_cat.drop(columns="n").add_suffix(" (Fine-Tuned)")
)

# Compute deltas for each metric
for metric in baseline_cat.columns.drop("n"):
    cat_comparison[f"{metric} (Delta)"] = cat_comparison[f"{metric} (Fine-Tuned)"] - cat_comparison[f"{metric} (Baseline)"]

# Reorder columns: group by metric [Baseline, Fine-Tuned, Delta]
ordered_cols = []
for metric in baseline_cat.columns.drop("n"):
    ordered_cols.extend([f"{metric} (Baseline)", f"{metric} (Fine-Tuned)", f"{metric} (Delta)"])

cat_comparison[ordered_cols]

,json_valid (Baseline),json_valid (Fine-Tuned),json_valid (Delta),func_name_correct (Baseline),func_name_correct (Fine-Tuned),func_name_correct (Delta),required_fields (Baseline),required_fields (Fine-Tuned),required_fields (Delta),type_correct (Baseline),...,enum_compliant (Delta),hallucinated_params (Baseline),hallucinated_params (Fine-Tuned),hallucinated_params (Delta),argument_match (Baseline),argument_match (Fine-Tuned),argument_match (Delta),no_tool_restraint (Baseline),no_tool_restraint (Fine-Tuned),no_tool_restraint (Delta)
category,,,,,,,,,,,,,,,,,,,,,
ambiguous,0.812500,1.0,0.1875,0.923077,0.9375,0.014423,0.923077,0.812500,-0.110577,1.0,...,0.000000,0.000000,0.0,0.000000,0.714469,0.704018,-0.010451,NaN,NaN,NaN
complex,0.837838,1.0,0.162162,0.903226,0.972973,0.069747,0.961290,0.972973,0.011683,1.0,...,-0.027027,0.000000,0.0,0.000000,0.652381,0.775933,0.123552,NaN,NaN,NaN
multi_tool,0.750000,0.875,0.125,0.833333,1.0,0.166667,0.877778,1.000000,0.122222,1.0,...,0.000000,0.055556,0.0,-0.055556,0.542284,0.784249,0.241965,NaN,NaN,NaN
no_tool,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.0
simple,0.951613,1.0,0.048387,0.949153,0.967742,0.018589,0.983051,0.967742,-0.015309,1.0,...,0.000000,0.000000,0.0,0.000000,0.910169,0.925000,0.014831,NaN,NaN,NaN


## Summary

Results will be filled after running the notebook.